# Module 6: Difference in Differences, as a Regression

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

[Module 5](Module_05_Difference_In_Differences_By_Hand.ipynb) produced 12.5
percent from four numbers. This module produces the same estimate from a
regression, which buys three things the hand calculation cannot:

- **an interval**, so the estimate can be judged
- **fixed effects**, which handle agency and month differences without naming them
- **a place to put everything else**, which the rest of the series needs

**About 25 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

keep = [a for a in TRAINED if a != "A007"]
d = f[f["agency_id"].isin(keep + COMPARISON)].copy()

d["lo"] = np.log(d["n_arrests"])                     # exposure, as an offset
d["tr"] = d["agency_id"].isin(keep).astype(float)
d["post"] = (d["period"] == "after").astype(float)
d["settled"] = ((d["tr"] == 1) & (d["period"] == "after")).astype(float)
d["phase"] = ((d["tr"] == 1) & (d["period"] == "phase")).astype(float)

pct = lambda b: 100 * (np.exp(b) - 1)


def fit(formula):
    return smf.glm(formula, d, family=sm.families.Poisson(), offset=d["lo"]).fit()


def report(name, z, term="settled"):
    lo, hi = z.conf_int().loc[term]
    print(f"  {name:34s} {pct(z.params[term]):+6.1f}%  "
          f"[{pct(lo):+6.1f}, {pct(hi):+6.1f}]   AIC {z.aic:,.0f}")


print(f"{len(d)} agency months, {d['agency_id'].nunique()} agencies")

## 2. Why a Poisson regression rather than least squares

The obvious move is to regress the log of the rate on a group indicator, a
period indicator and their interaction. It does not work here, for a reason
worth seeing.

In [ ]:
zeros = int((d["n_uof"] == 0).sum())
print(f"  agency months with zero use of force: {zeros} of {len(d)}"
      f"  ({100 * zeros / len(d):.0f} percent)")
print(f"  log of zero is {np.log(0)}, so those rows are lost or the fit fails")

Ten percent of the rows would be dropped or would break the fit, and the
dropped rows are not random: they are the smallest agencies and the quietest
months.

A **Poisson regression with the log of arrests as an offset** models the count
directly, handles zeros without complaint, and produces coefficients that are
already proportional changes. That is the default for this kind of data, and
Time Series Advanced [Module 8](../../../Time_Series/Advanced/Module_08_Count_Regression_With_Harmonics.md)
sets out why.

## 3. The simplest specification reproduces the hand calculation

In [ ]:
z1 = fit("n_uof ~ tr + post + settled + phase")
report("group and period indicators", z1)
print(f"\n  the hand calculation in Module 5 gave  -12.5%")
print(f"  the truth is                           {TRUTH:+.1f}%")

The same number, to within a tenth of a point, and now with an interval from
17.9 to 6.9 percent below. The small gap against the hand calculation is
because the regression uses one row per agency month while the hand version
summed the four cells first.

**The interval is the point of this module.** The hand calculation could not
tell you that an effect of 7 percent and an effect of 18 percent are both
consistent with these records.

## 4. Fixed effects, and what they buy

`C(agency_id)` gives each agency its own baseline. `C(year_month)` gives each
month its own level, shared across agencies.

In [ ]:
z2 = fit("n_uof ~ C(agency_id) + post + settled + phase")
z3 = fit("n_uof ~ C(agency_id) + C(year_month) + settled + phase")
report("group and period indicators", z1)
report("agency fixed effects", z2)
report("agency and month fixed effects", z3)

Two things are true at once and both are worth saying.

**The estimate barely moves**, from 12.6 to 12.5 to 12.6 percent, and the
interval barely moves either. The simple specification was not wrong.

**The fit improves enormously**, with AIC falling by more than 400 points. The
month effects are capturing real structure, the seasonal pattern and the
statewide trend, that the single `post` indicator was approximating with one
number.

The reason the estimate did not move is that `post` was already doing the
essential job: absorbing whatever happened to both groups between the two
periods. Month effects do it flexibly rather than with one step, which fits
better without changing the comparison.

**Do not conclude that fixed effects never matter.** Drop the `post` term from
the second specification and watch what happens.

In [ ]:
z_bad = fit("n_uof ~ C(agency_id) + settled + phase")
report("agency effects, no time term at all", z_bad)
print(f"\n  the truth is {TRUTH:+.1f}%")

Without anything absorbing time, the settled indicator picks up four years of
statewide decline on top of the program, and the estimate more than doubles.

**Something must account for time.** A post indicator is the minimum; month
effects are the safe default.

## 5. Reading the rest of the output

The coefficients you did not ask for are worth a look, and Time Series
Advanced [Module 10](../../../Time_Series/Advanced/Module_10_Panel_And_Hierarchical.md)
makes the case that they are the best misspecification detector available.

In [ ]:
lo, hi = z3.conf_int().loc["phase"]
print(f"  the phase in coefficient: {pct(z3.params['phase']):+.1f}%  "
      f"[{pct(lo):+.1f}, {pct(hi):+.1f}]")
print(f"  the settled coefficient:  {pct(z3.params['settled']):+.1f}%")
print("\n  the phase in should sit between zero and the settled effect")
print(f"  does it?  {TRUTH < pct(z3.params['phase']) < 0}")
print(f"\n  Pearson dispersion: {z3.pearson_chi2 / z3.df_resid:.2f}   (1.0 is Poisson)")

The phase in coefficient lands between zero and the full effect, which is what
a partially implemented program should look like. If it had come out
**positive**, as it does when the time structure is missing, that would be a
signal to go back rather than a finding about the program.

## Exercise

The model above pools all four trained agencies into one coefficient. Let each
have its own and see whether they differ by more than noise.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    d2 = d.copy()
    for a in keep:
        d2[f"s_{a}"] = ((d2["agency_id"] == a) & (d2["period"] == "after")).astype(float)
    terms = " + ".join(f"s_{a}" for a in keep)
    z = smf.glm(f"n_uof ~ C(agency_id) + C(year_month) + phase + {terms}",
                d2, family=sm.families.Poisson(), offset=d2["lo"]).fit()
    rows = []
    for a in keep:
        lo, hi = z.conf_int().loc[f"s_{a}"]
        rows.append({"agency": NAME[a], "estimate": f"{pct(z.params[f's_{a}']):+.1f}%",
                     "95 percent interval": f"[{pct(lo):+.1f}, {pct(hi):+.1f}]"})
    lo, hi = z3.conf_int().loc["settled"]
    rows.append({"agency": "ALL FOUR POOLED",
                 "estimate": f"{pct(z3.params['settled']):+.1f}%",
                 "95 percent interval": f"[{pct(lo):+.1f}, {pct(hi):+.1f}]"})
    print(f"  the truth is {TRUTH:+.1f} percent at every one of them\n")
    display(pd.DataFrame(rows).set_index("agency"))
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

The four per agency intervals all overlap each other and all contain the
pooled estimate. There is no evidence the effect differed between agencies,
and in fact it did not: the planted effect is identical at all four.

**Note how wide the individual intervals are.** They run from 13 points at
Stonewick to 40 at Pinecrest, averaging 25, against 11 for the pooled
estimate. That is the cost of asking four questions instead of one, and it is
why a report should lead with the pooled number.

The two widest belong to the two smallest agencies, which is the pattern from
Time Series Advanced Module 9: what an agency can detect is set by how many
incidents it records.

The temptation this exercise is guarding against is real. Given four
estimates, a reader will rank them and ask why the program worked best
somewhere. The answer here is that it did not, and the intervals say so.
Module 15 makes that check formal.

</details>

---

**Next:** [Module 7: Testing Parallel Trends](Module_07_Testing_Parallel_Trends.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*